# Notebook 14R — A-noncausal-uniform vs A-causal-uniform (compute-matched, 3 seeds)

Mục tiêu duy nhất của notebook này là trả lời **RQ-A: loại future edges khỏi sampled neighborhood có làm thay đổi hiệu năng FraudGT không?**

Hai nhánh giữ nguyên kiến trúc, dữ liệu, split, seed, optimizer, loss, epoch, fanout và effective batch size. Chúng chỉ khác sampler:

- **A-noncausal-uniform-clean:** `clean_link_neighbor`, lấy mẫu uniform và cho phép history có future edges, nhưng luôn tách target transaction khỏi neighborhood rồi gắn lại đúng một lần;
- **A-causal-uniform-clean:** `temporal_link_neighbor`, `temporal_strategy=uniform`, `temporal_strict=True`, chỉ lấy mẫu uniform trong các cạnh có thời gian nhỏ hơn target.

Protocol clean: chuẩn hóa bằng thống kê train, validation/test không shuffle, đánh giá toàn bộ split, chọn epoch và threshold bằng validation, bổ sung PR-AUC (`ap`). Chế độ full chạy ba seed 42–44 và báo cáo mean ± std.

**Ngân sách train:** batch 256 × accumulation 8 × 200 epoch có cùng effective batch, số target samples và số optimizer updates với cấu hình cũ batch 512 × accumulation 4 × 100 epoch. Full validation/test chạy mỗi 25 epoch để vừa giới hạn phiên. **Thời gian dự kiến trên Kaggle 2×T4:** khoảng 9–11 giờ khi hai nhánh chạy song song. Notebook lưu recovery checkpoint mỗi 25 epoch.

In [ ]:
SEED = 42
REPEATS = 3  # fraudGT.main dùng seed 42, 43, 44
RUN_MODE = 'full'  # 3 seeds, compute-matched training, full validation/test
MAX_EPOCHS = 200
TRAIN_STEPS = 256
EVAL_PERIOD = 25
BATCH_SIZE = 256
BATCH_ACCUMULATION = 8  # effective batch = 2048
WARMUP_EPOCHS = 10
FANOUT = [25, 25]
NUM_THREADS = 2
NUM_WORKERS = 2
assert RUN_MODE in {'smoke', 'pilot', 'full'}
if RUN_MODE == 'full': assert REPEATS == 3
if RUN_MODE == 'full':
    assert MAX_EPOCHS * TRAIN_STEPS * BATCH_SIZE == 100 * 256 * 512
    assert MAX_EPOCHS * TRAIN_STEPS // BATCH_ACCUMULATION == 100 * 256 // 4
print('Mode:', RUN_MODE, '| seed:', SEED, '| effective batch:', BATCH_SIZE * BATCH_ACCUMULATION)
print('Target exposures/model:', MAX_EPOCHS * TRAIN_STEPS * BATCH_SIZE, '| optimizer updates/model:', MAX_EPOCHS * TRAIN_STEPS // BATCH_ACCUMULATION)

## 1. Môi trường và dependencies

In [ ]:
import importlib, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path
import torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())
if RUN_MODE == 'full' and torch.cuda.device_count() < 2:
    raise RuntimeError('Full 3-seed yêu cầu Kaggle 2×T4; một GPU có nguy cơ vượt giới hạn phiên.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
compiled_modules = ['pyg_lib', 'torch_scatter', 'torch_sparse']
def missing_modules():
    importlib.invalidate_caches()
    missing = []
    for name in compiled_modules:
        try:
            importlib.import_module(name)
        except (ImportError, OSError):
            missing.append(name)
    return missing

missing = missing_modules()
if missing:
    print('Missing compiled modules:', missing)
    try:
        print('Trying pre-built wheels:', wheel_url)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--retries', '1', '--timeout', '10',
                        'pyg_lib', 'torch_scatter', 'torch_sparse',
                        '-f', wheel_url], check=True)
    except subprocess.CalledProcessError:
        print('data.pyg.org unavailable; building official sources instead.')
        if torch.version.cuda and shutil.which('nvcc') is None:
            raise RuntimeError('nvcc is unavailable. Use a Kaggle GPU image with CUDA toolkit or attach pre-built wheels as a Dataset.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'ninja', 'wheel', 'cmake', 'setuptools'], check=True)
        build_env = os.environ.copy()
        build_env['MAX_JOBS'] = '2'
        build_env['TORCH_CUDA_ARCH_LIST'] = '7.5'  # Tesla T4
        if Path('/usr/local/cuda').exists():
            build_env['CUDA_HOME'] = '/usr/local/cuda'

        source_root = Path('/kaggle/working/pyg_source_build')
        shutil.rmtree(source_root, ignore_errors=True)
        source_root.mkdir(parents=True)

        def fetch_github_archive(owner, repository, ref, destination):
            url = f'https://github.com/{owner}/{repository}/archive/{ref}.tar.gz'
            archive = source_root / f'{repository}-{ref[:12]}.tar.gz'
            print('Downloading:', url)
            with urllib.request.urlopen(url, timeout=180) as response, archive.open('wb') as out:
                shutil.copyfileobj(response, out)
            shutil.rmtree(destination, ignore_errors=True)
            destination.mkdir(parents=True)
            with tarfile.open(archive, 'r:gz') as tf:
                members = []
                for member in tf.getmembers():
                    parts = Path(member.name).parts[1:]  # remove archive root directory
                    if not parts:
                        continue
                    member.name = str(Path(*parts))
                    members.append(member)
                tf.extractall(destination, members=members, filter='data')
            return destination

        # pyg-lib 0.8.0 is the release supporting PyTorch 2.10.
        # GitHub source archives omit git submodules, so restore every
        # dependency needed by the CPU neighbor sampler explicitly.
        pyg_src = fetch_github_archive('pyg-team', 'pyg-lib', '0.8.0', source_root / 'pyg-lib')
        fetch_github_archive('greg7mdp', 'parallel-hashmap',
                             '2ec799017610ef831f4dc29c21fb3cce7e4a19b9',
                             pyg_src / 'third_party' / 'parallel-hashmap')
        fetch_github_archive('KarypisLab', 'METIS',
                             '22008804e8c9b78893ae10a94c0d8b4b592438b4',
                             pyg_src / 'third_party' / 'METIS')
        # METIS itself pins GKlib as a nested git submodule. Its archive
        # also omits that directory, so restore the exact pinned commit.
        fetch_github_archive('KarypisLab', 'GKlib',
                             '3eabb216ac97e11ce7e7a9b90f4c90778d9e7c18',
                             pyg_src / 'third_party' / 'METIS' / 'GKlib')
        required_source_files = [
            pyg_src / 'third_party' / 'parallel-hashmap' / 'parallel_hashmap' / 'phmap.h',
            pyg_src / 'third_party' / 'METIS' / 'include' / 'metis.h',
            pyg_src / 'third_party' / 'METIS' / 'GKlib' / 'GKlibSystem.cmake',
        ]
        missing_source_files = [str(path) for path in required_source_files if not path.exists()]
        if missing_source_files:
            raise RuntimeError('Incomplete pyg-lib source tree: ' + ', '.join(missing_source_files))
        pyg_env = build_env.copy()
        pyg_env['FORCE_CUDA'] = '0'  # sampling is CPU-side; avoids a large CUDA build
        pyg_env['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
        print('Building pyg-lib 0.8.0 CPU sampler backend...')
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        '--no-build-isolation', str(pyg_src)], check=True, env=pyg_env)

        scatter_src = fetch_github_archive('rusty1s', 'pytorch_scatter', '2.1.2',
                                             source_root / 'pytorch_scatter')
        print('Building torch-scatter 2.1.2 for CUDA T4...')
        cuda_env = build_env.copy()
        cuda_env['FORCE_CUDA'] = '1'
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        '--no-build-isolation', str(scatter_src)], check=True, env=cuda_env)

        sparse_src = fetch_github_archive('rusty1s', 'pytorch_sparse', '0.6.18',
                                            source_root / 'pytorch_sparse')
        fetch_github_archive('greg7mdp', 'parallel-hashmap',
                             '01ea8093e6d0293ea252e8027c17d7dff26a9c9f',
                             sparse_src / 'third_party' / 'parallel-hashmap')
        print('Building torch-sparse 0.6.18 for CUDA T4...')
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        '--no-build-isolation', str(sparse_src)], check=True, env=cuda_env)

remaining = missing_modules()
if remaining:
    raise RuntimeError(f'PyG extensions are still unavailable: {remaining}')
print('Compiled PyG extensions imported successfully.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'torchmetrics', 'yacs', 'datatable', 'pandas', 'matplotlib', 'seaborn', 'wandb', 'ogb', 'tensorboardX', 'pyyaml', 'pytest'], check=True)
print('Dependencies installed.')

## 2. Lấy đúng source và dữ liệu AML-Small-HI

In [ ]:
from shutil import copy2
REPO_URL = 'https://github.com/mhiunguyen/TH-FraudGT.git'
repo = Path('/kaggle/working/TH-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', commit)
required_markers = {
    repo / 'fraudGT/datasets/aml_dataset.py': 'Fit preprocessing statistics on train only',
    repo / 'fraudGT/logger.py': "'ap': reformat(average_precision_score",
    repo / 'fraudGT/sampler/custom_sampler.py': "@register_sampler('clean_link_neighbor')",
}
for path, marker in required_markers.items():
    if not path.exists() or marker not in path.read_text(encoding='utf-8'):
        raise RuntimeError(f'Source trên Git chưa có clean protocol: {path.name} thiếu marker {marker!r}. Hãy push bản Notebook 14 trước khi chạy.')
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_temporal_target_batch.py'], cwd=repo, check=True)
print('PASS temporal target-batch unit tests.')
candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Hãy Add Input bộ IBM AML; thiếu HI-Small_Trans.csv.')
destination = repo / 'data/AML/HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != candidates[0].stat().st_size:
    copy2(candidates[0], destination)
print('Dataset:', destination, '| MiB:', round(destination.stat().st_size / 1024**2, 1))

## 3. Unit audit: temporal sampler phải strict-past

In [ ]:
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
toy = HeteroData()
toy['node'].x = torch.ones(5, 1); toy['node'].num_nodes = 5
edge_index = torch.tensor([[0, 1, 2, 0, 3], [1, 2, 3, 3, 4]])
edge_time = torch.tensor([1, 2, 5, 5, 6], dtype=torch.long)
for relation, index in [(('node','to','node'), edge_index), (('node','rev_to','node'), edge_index.flip(0))]:
    toy[relation].edge_index = index
    toy[relation].edge_attr = torch.ones(edge_index.shape[1], 1)
    toy[relation].temporal_timestamps = edge_time
target_time = torch.tensor([5], dtype=torch.long)
audit_loader = LinkNeighborLoader(toy, num_neighbors=[-1, -1], edge_label_index=(('node','to','node'), torch.tensor([[0], [3]])), edge_label=torch.tensor([0]), edge_label_time=target_time - 1, time_attr='temporal_timestamps', temporal_strategy='uniform', disjoint=True, batch_size=1, shuffle=False)
audit_batch = next(iter(audit_loader))
sampled_times = []
for relation in audit_batch.edge_types:
    sampled_times.extend(audit_batch[relation].temporal_timestamps.tolist())
assert sampled_times and max(sampled_times) < target_time.item(), sampled_times
print('PASS strict-past audit | sampled timestamps:', sorted(set(sampled_times)))

## 4. Sinh hai config công bằng
Cell này tạo cả hai config từ **cùng một base**, sau đó chỉ thay sampler. `val.iter_per_epoch=-1` bắt buộc đánh giá toàn bộ validation/test split.

In [ ]:
import copy, yaml
base_path = repo / 'configs/AML-Small-HI/AML-Small-HI-A-Retrain-T4.yaml'
base = yaml.safe_load(base_path.read_text(encoding='utf-8'))
epochs = 2 if RUN_MODE == 'smoke' else MAX_EPOCHS
train_steps = 4 if RUN_MODE == 'smoke' else TRAIN_STEPS
eval_period = 1 if RUN_MODE == 'smoke' else EVAL_PERIOD
common = copy.deepcopy(base)
common['out_dir'] = str(repo / 'results_nb14r_uniform')
common['seed'] = SEED
common['dataset']['dir'] = str(repo / 'data')
common['dataset']['add_history'] = False
common['num_threads'] = NUM_THREADS; common['num_workers'] = NUM_WORKERS
common['wandb']['use'] = False
common['train']['neighbor_sizes'] = FANOUT
common['train']['batch_size'] = BATCH_SIZE
common['train']['iter_per_epoch'] = train_steps
common['train']['eval_period'] = eval_period
common['train']['persistent_workers'] = False
common['train']['enable_ckpt'] = True
common['train']['ckpt_best'] = True
common['train']['auto_resume'] = True
common['train']['ckpt_resume_period'] = EVAL_PERIOD
common['val']['iter_per_epoch'] = 4 if RUN_MODE == 'smoke' else -1
common['optim']['max_epoch'] = epochs
common['optim']['batch_accumulation'] = BATCH_ACCUMULATION
common['optim']['num_warmup_epochs'] = 1 if RUN_MODE == 'smoke' else WARMUP_EPOCHS
common['mvia']['thresholds'] = [round(x / 100, 2) for x in range(5, 100, 5)]
noncausal = copy.deepcopy(common)
noncausal['train']['sampler'] = 'clean_link_neighbor'; noncausal['val']['sampler'] = 'clean_link_neighbor'
causal = copy.deepcopy(common)
causal['train']['sampler'] = 'temporal_link_neighbor'; causal['val']['sampler'] = 'temporal_link_neighbor'
causal['train']['temporal_strategy'] = 'uniform'; causal['train']['temporal_strict'] = True
# Fairness guard: sau khi bỏ đúng ba temporal fields, hai config phải giống nhau.
fair_causal = copy.deepcopy(causal)
fair_causal['train']['sampler'] = 'clean_link_neighbor'; fair_causal['val']['sampler'] = 'clean_link_neighbor'
fair_causal['train'].pop('temporal_strategy'); fair_causal['train'].pop('temporal_strict')
assert fair_causal == noncausal, 'Hai config còn khác ngoài causal sampler.'
generated = Path('/kaggle/working/generated_configs_nb14r_uniform'); generated.mkdir(exist_ok=True)
NC_CFG = generated / 'AML-Small-HI-A14R-noncausal-uniform-clean.yaml'
C_CFG = generated / 'AML-Small-HI-A14R-causal-uniform-clean.yaml'
NC_CFG.write_text(yaml.safe_dump(noncausal, sort_keys=False), encoding='utf-8')
C_CFG.write_text(yaml.safe_dump(causal, sort_keys=False), encoding='utf-8')
assert yaml.safe_load(NC_CFG.read_text())['train']['sampler'] == 'clean_link_neighbor'
assert yaml.safe_load(C_CFG.read_text())['train']['sampler'] == 'temporal_link_neighbor'
assert yaml.safe_load(C_CFG.read_text())['train']['temporal_strategy'] == 'uniform'
print('Noncausal config:', NC_CFG, '| sampler: clean_link_neighbor | strategy: uniform')
print('Causal config:', C_CFG, '| sampler: temporal_link_neighbor | strategy: uniform | strict past')
print('epochs:', epochs, '| repeats:', REPEATS, '| train steps:', train_steps, '| full eval:', RUN_MODE != 'smoke')

## 5. Tạo lại cache theo clean normalization và kiểm tra số batch

In [ ]:
import gc, math
# Cache cũ có normalization theo từng split, vì vậy phải xóa đúng hai file cache AML-Small-HI.
processed = repo / 'data/AML/Small-HI/processed'
for filename in ['data.pt', 'ports.pt']:
    path = processed / filename
    if path.exists():
        path.unlink(); print('Removed stale cache:', path)
sys.path.insert(0, str(repo))
from fraudGT.datasets.aml_dataset import AMLDataset
started = time.time()
dataset = AMLDataset(root=str(repo / 'data/AML'), name='Small-HI', reverse_mp=True, add_ports=True, add_history=False)
task = ('node','to','node')
counts = {split: int(dataset[split][task].split_mask.sum()) for split in ['train','val','test']}
batches = {split: math.ceil(count / BATCH_SIZE) for split, count in counts.items()}
print('Target edges:', counts)
print('Full-split batches:', batches)
print(f'Clean cache ready in {(time.time()-started)/60:.1f} min')
del dataset; gc.collect()
if RUN_MODE in {'pilot', 'full'} and batches['test'] > 5000:
    print('INFO: test có hơn 5,000 batch; compute-matched 3-seed cần khoảng 9–11 giờ trên 2×T4.')

## 6. Huấn luyện song song trên 2 GPU
Không mở notebook GPU khác trong lúc chạy. Heartbeat mỗi 60 giây không phải lỗi.

In [ ]:
jobs = [
    {'name':'A-noncausal-uniform-clean', 'cfg':NC_CFG, 'gpu':0, 'tag':f'NB14R-{RUN_MODE}', 'log':Path('/kaggle/working/NB14R_noncausal_uniform.log')},
    {'name':'A-causal-uniform-clean', 'cfg':C_CFG, 'gpu':1 if torch.cuda.device_count() >= 2 else 0, 'tag':f'NB14R-{RUN_MODE}', 'log':Path('/kaggle/working/NB14R_causal_uniform.log')},
]
def command(job):
    return [sys.executable, '-u', '-m', 'fraudGT.main', '--cfg', str(job['cfg']), '--repeat', str(REPEATS), '--gpu', str(job['gpu']), 'name_tag', job['tag']]
def run_jobs(selected):
    handles = []
    for job in selected:
        stream = job['log'].open('w', encoding='utf-8')
        process = subprocess.Popen(command(job), cwd=repo, stdout=stream, stderr=subprocess.STDOUT, text=True)
        handles.append((job, process, stream)); print(f"Started {job['name']} on GPU {job['gpu']} | PID {process.pid}")
    started = time.time()
    while any(process.poll() is None for _, process, _ in handles):
        time.sleep(60)
        print(f'[heartbeat] {(time.time()-started)/60:.0f} min', [(job['name'], 'running' if process.poll() is None else 'done') for job,process,_ in handles], flush=True)
        subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,utilization.gpu','--format=csv,noheader'], check=False)
    failed = []
    for job, process, stream in handles:
        stream.close()
        if process.returncode != 0:
            failed.append(job['name']); print('\n'.join(job['log'].read_text(errors='replace').splitlines()[-100:]))
    if failed: raise RuntimeError('Failed: ' + ', '.join(failed))
if torch.cuda.device_count() >= 2:
    run_jobs(jobs)
else:
    print('Chỉ có 1 GPU: hai mô hình sẽ chạy tuần tự.')
    for job in jobs: run_jobs([job])
for job in jobs:
    text = job['log'].read_text(encoding='utf-8', errors='replace').lower()
    if "loss': nan" in text or 'contains nan/inf' in text or 'no supervision' in text:
        raise RuntimeError(f"{job['name']} failed numerical/supervision audit. Không được dùng kết quả.")
print('PASS: training complete, no empty supervision and no NaN/Inf.')

## 7. Chọn epoch/threshold bằng validation và báo cáo test
Báo cáo cả threshold cố định 0.10 và threshold được chọn trên validation. Không chọn theo test.

In [ ]:
import pandas as pd
summarizer = repo / 'scripts/summarize_thresholds.py'
frames = []; evidence = [NC_CFG, C_CFG]
for job in jobs:
    run_dir = repo / 'results_nb14r_uniform' / f"{job['cfg'].stem}-{job['tag']}-gpu{job['gpu']}"
    for protocol, extra in [('fixed_0.10', ['--fixed-threshold','0.10']), ('validation_selected', [])]:
        output = Path(f"/kaggle/working/summary_NB14R_{job['name']}_{protocol}.csv")
        subprocess.run([sys.executable, str(summarizer), str(run_dir), '--output', str(output)] + extra, check=True)
        frame = pd.read_csv(output); frame.insert(0, 'model', job['name']); frame.insert(1, 'protocol', protocol)
        frames.append(frame); evidence.extend([output, job['log']])
results = pd.concat(frames, ignore_index=True)
expected_seeds = set(range(SEED, SEED + REPEATS))
for model in results['model'].unique():
    actual = set(results.loc[(results['model'] == model) & (results['protocol'] == 'validation_selected'), 'seed'])
    assert actual == expected_seeds, f'{model}: expected seeds {expected_seeds}, got {actual}'
RESULTS = Path('/kaggle/working/summary_NB14R_uniform_causal_impact_3seeds.csv')
results.to_csv(RESULTS, index=False); evidence.append(RESULTS)
display(results[['model','protocol','seed','best_epoch','threshold','val_f1','test_f1','test_precision','test_recall','test_auc','test_ap']])
print('Summary:', RESULTS)

## 8. Biểu đồ so sánh và cách đọc kết quả

In [ ]:
import matplotlib.pyplot as plt
plot_df = results[results['protocol'] == 'validation_selected'].copy()
metrics = ['test_f1','test_precision','test_recall','test_ap','test_auc']
summary_metrics = ['best_epoch','threshold','val_f1'] + metrics
aggregate = results.groupby(['model','protocol'])[summary_metrics].agg(['mean','std']).round(5)
AGGREGATE = Path('/kaggle/working/summary_NB14R_uniform_causal_impact_3seeds_mean_std.csv')
aggregate.to_csv(AGGREGATE); evidence.append(AGGREGATE)
display(aggregate)
model_order = ['A-noncausal-uniform-clean', 'A-causal-uniform-clean']
fig, axes = plt.subplots(1, len(metrics), figsize=(19, 4))
for ax, metric in zip(axes, metrics):
    grouped = plot_df.groupby('model')[metric].agg(['mean','std']).reindex(model_order)
    ax.bar(model_order, grouped['mean'], yerr=grouped['std'], capsize=5, color=['#4472C4','#ED7D31'])
    ax.set_title(metric); ax.tick_params(axis='x', rotation=25); ax.grid(axis='y', alpha=.25)
fig.suptitle('Notebook 14R — uniform causal impact, seeds 42–44 (compute-matched)'); fig.tight_layout()
FIGURE = Path('/kaggle/working/NB14R_uniform_causal_impact_3seeds.png')
fig.savefig(FIGURE, dpi=180, bbox_inches='tight'); evidence.append(FIGURE)
plt.show()
display(pd.DataFrame([
    ['causal ≈ noncausal', 'Future edges tồn tại nhưng chưa cho thấy tác động metric ổn định qua ba seed.'],
    ['causal < noncausal', 'Baseline cũ có bằng chứng hưởng lợi từ future information.'],
    ['causal > noncausal', 'Future edges có thể gây nhiễu; causal sampling là kết quả đáng chú ý.'],
    ['dao động/khó đọc', 'Kiểm tra log, full split, AP và chạy thêm seed trước khi diễn giải.'],
], columns=['Kết quả', 'Diễn giải hợp lệ']))

## 9. Giới hạn và quyết định tiếp theo

Notebook 14R kiểm định **tác động của strict-past cutoff** khi cả hai nhánh cùng sampling strategy `uniform` và cùng compute budget. Nó không tự động chứng minh coverage thấp làm dự đoán kém. Sampling audit ở notebook 13 là bằng chứng cấu trúc riêng.

- Dùng mean ± std của ba seed để kết luận, không chọn riêng seed đẹp.
- Sau bước này: phân tích prediction theo activity/coverage hoặc so sánh H-causal với A-causal.
- Chỉ báo cáo test tại epoch/threshold đã chọn bằng validation.

## 10. Đóng gói artifact để tải về

In [ ]:
import shutil
bundle = Path('/kaggle/working/NB14R_uniform_causal_impact_3seeds_artifacts'); bundle.mkdir(exist_ok=True)
for path in dict.fromkeys(map(Path, evidence)):
    if path.exists(): shutil.copy2(path, bundle / path.name)
(bundle / 'commit.txt').write_text(commit + '\n', encoding='utf-8')
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('Download:', archive)

# Trong lúc train nên học gì?

Ưu tiên theo thứ tự, không đọc lan man:

1. **Pipeline FraudGT gốc:** target transaction → link-neighbor sampling → RMP/Ports/Ego ID → FraudGT Encoder → edge head.
2. **Temporal leakage:** phân biệt split theo thời gian với causal neighborhood; hiểu điều kiện `neighbor_time < target_time`.
3. **Đánh giá dữ liệu mất cân bằng:** precision, recall, F1, ROC-AUC và đặc biệt PR-AUC/AP; hiểu tại sao accuracy gần 99.9% vẫn có thể vô nghĩa.
4. **Validation protocol:** validation dùng chọn epoch/threshold; test chỉ báo cáo cuối; một seed là pilot, ba seed mới dùng để so sánh.
5. **Đọc lại notebook 13:** nắm ba số của chính đồ án — history coverage, pair-history coverage và future-edge ratio.

Khi train xong, trước tiên xem `summary_NB14R_uniform_causal_impact_3seeds_mean_std.csv`, sau đó mới xem log. Đừng chỉ nhìn F1; phải xem cùng PR-AUC, precision và recall.